# Mini-Project 3: Lake Victoria Fish Stock & Export Risk Model
## Assumptions
- Stock is measured in tonnes and simulated weekly for 52 weeks.
- The harvest fraction `h` is applied to the stock present at the start of each week: harvest = `h * N(t)`.
- Price is in UGX/kg and follows a seeded random walk, clipped to UGX 9,000–16,000/kg.
- Revenue is calculated as weekly harvest in kg multiplied by weekly price in UGX/kg.
- Risk classes use the coefficient of variation (CV): Low < 10%, Moderate 10%–20%, High > 20%. These are transparent illustrative thresholds, not official industry standards.
- A fixed seed makes the analysis reproducible.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from statistics import mean, median, variance, stdev
from typing import Iterable

import numpy as np
import matplotlib.pyplot as plt

SEED = 2026
R = 0.4
K = 10_000.0
N0 = 4_000.0
WEEKS = 52
PRICE_START = 12_000.0
PRICE_MIN = 9_000.0
PRICE_MAX = 16_000.0

np.set_printoptions(precision=2, suppress=True)
print("NumPy version:", np.__version__)

## 1. Fibonacci baseline and its limitations

The first 15 Fibonacci numbers are shown as the requested baseline. They are **not** used as the biological stock model.

In [ ]:
def fibonacci_numbers(count: int) -> list[int]:
    """Return the first `count` Fibonacci numbers, starting with 0, 1."""
    if count < 0:
        raise ValueError("count must be non-negative")
    values: list[int] = []
    a, b = 0, 1
    for _ in range(count):
        values.append(a)
        a, b = b, a + b
    return values

fib_stock = fibonacci_numbers(15)
print("First 15 Fibonacci numbers:", fib_stock)

Fibonacci numbers grow without reference to the lake's carrying capacity, food supply, habitat, recruitment, disease, or fishing pressure. A real fish population cannot increase indefinitely because resources and suitable habitat are finite. The sequence also has no meaningful calibration to tonnes of fish or to a weekly biological growth rate. It is therefore useful only as a toy numerical baseline, not as a defensible stock or harvest forecast.

## 2. Object-oriented fish stock model

The discrete logistic model with proportional harvesting is

\[
N_{t+1}=N_t+rN_t\left(1-\frac{N_t}{K}\right)-hN_t.
\]

The implementation validates parameters, stores the stock trajectory, and calculates weekly harvest.

In [ ]:
@dataclass
class FishStock:

    initial_stock: float = N0
    growth_rate: float = R
    carrying_capacity: float = K
    harvest_rate: float = 0.10

    def __post_init__(self) -> None:
        if self.initial_stock < 0:
            raise ValueError("initial_stock must be non-negative")
        if self.carrying_capacity <= 0:
            raise ValueError("carrying_capacity must be positive")
        if not 0 <= self.initial_stock <= self.carrying_capacity:
            raise ValueError("initial_stock must not exceed carrying_capacity")
        if self.growth_rate < 0:
            raise ValueError("growth_rate must be non-negative")
        if not 0 <= self.harvest_rate <= 1:
            raise ValueError("harvest_rate must be between 0 and 1")

    def simulate(self, weeks: int = WEEKS) -> np.ndarray:
        """Return stock at weeks 0..weeks, in tonnes."""
        if weeks < 0:
            raise ValueError("weeks must be non-negative")
        stock = np.zeros(weeks + 1, dtype=float)
        stock[0] = self.initial_stock
        for t in range(weeks):
            n = stock[t]
            next_n = n + self.growth_rate * n * (1 - n / self.carrying_capacity) - self.harvest_rate * n
            stock[t + 1] = np.clip(next_n, 0, self.carrying_capacity)
        return stock

    def weekly_harvest(self, stock: np.ndarray) -> np.ndarray:
        """Compute harvest in tonnes for each week from stock at the week's start."""
        stock = np.asarray(stock, dtype=float)
        if stock.ndim != 1 or stock.size < 1:
            raise ValueError("stock must be a non-empty 1D array")
        if np.any(stock < 0):
            raise ValueError("stock values must be non-negative")
        return self.harvest_rate * stock[:-1]

    def simulate_with_closed_season(self, years: int = 5, closed_weeks: int = 8) -> tuple[np.ndarray, np.ndarray]:
        """Simulate several years with no harvest during the first `closed_weeks` of each year.

        Returns (stock trajectory in tonnes, weekly harvest in tonnes).
        The same logistic growth rule applies during closed weeks, but harvest is zero.
        """
        if years < 1 or not 0 <= closed_weeks <= WEEKS:
            raise ValueError("years must be >= 1 and closed_weeks must be between 0 and 52")
        total_weeks = years * WEEKS
        stock = np.zeros(total_weeks + 1, dtype=float)
        harvest = np.zeros(total_weeks, dtype=float)
        stock[0] = self.initial_stock
        for t in range(total_weeks):
            week_of_year = t % WEEKS
            h_t = 0.0 if week_of_year < closed_weeks else self.harvest_rate
            n = stock[t]
            harvest[t] = h_t * n
            next_n = n + self.growth_rate * n * (1 - n / self.carrying_capacity) - harvest[t]
            stock[t + 1] = np.clip(next_n, 0, self.carrying_capacity)
        return stock, harvest

baseline = FishStock(harvest_rate=0.10)
stock_10 = baseline.simulate()
harvest_10 = baseline.weekly_harvest(stock_10)
print(f"Stock at week 0: {stock_10[0]:,.1f} tonnes")
print(f"Stock at week 52: {stock_10[-1]:,.1f} tonnes")
print(f"Total harvest in 52 weeks: {harvest_10.sum():,.1f} tonnes")

## 3. Seeded weekly price model

A random walk is used because the task requests one. The random walk is clipped to the stated price bounds. This is a simple illustrative assumption and does not claim to represent observed fish-market dynamics.

In [ ]:
@dataclass
class PriceModel:
    """Generate reproducible weekly prices in UGX/kg using a bounded random walk."""
    start_price: float = PRICE_START
    lower_bound: float = PRICE_MIN
    upper_bound: float = PRICE_MAX
    weekly_shock_sd: float = 500.0
    seed: int = SEED

    def __post_init__(self) -> None:
        if self.lower_bound <= 0 or self.upper_bound <= self.lower_bound:
            raise ValueError("Price bounds must be positive and upper_bound > lower_bound")
        if not self.lower_bound <= self.start_price <= self.upper_bound:
            raise ValueError("start_price must be within price bounds")
        if self.weekly_shock_sd < 0:
            raise ValueError("weekly_shock_sd must be non-negative")

    def simulate(self, weeks: int = WEEKS, seed: int | None = None) -> np.ndarray:
        """Return `weeks` prices in UGX/kg, including the starting price."""
        if weeks < 1:
            raise ValueError("weeks must be at least 1")
        rng = np.random.default_rng(self.seed if seed is None else seed)
        prices = np.empty(weeks, dtype=float)
        prices[0] = self.start_price
        shocks = rng.normal(0, self.weekly_shock_sd, size=weeks - 1)
        for i, shock in enumerate(shocks, start=1):
            prices[i] = np.clip(prices[i - 1] + shock, self.lower_bound, self.upper_bound)
        return prices

price_model = PriceModel()
prices = price_model.simulate()
print("First 10 weekly prices (UGX/kg):", prices[:10])

## 4. Revenue statistics and why a raw variance threshold is flawed

Annual revenue is the sum of weekly revenue. Weekly harvest is converted from tonnes to kilograms before multiplying by price in UGX/kg. Variance has **squared currency units** (UGX²), so a threshold such as 50,000 has no clear meaning without a scale, a specified revenue unit, and a decision rationale. CV = standard deviation / mean is dimensionless and makes relative variability easier to compare, though it still needs a justified threshold.

In [ ]:
def revenue_from_harvest(harvest_tonnes: np.ndarray, prices_ugx_per_kg: np.ndarray) -> np.ndarray:
    """Calculate weekly revenue in UGX from tonnes harvested and UGX/kg prices."""
    harvest_tonnes = np.asarray(harvest_tonnes, dtype=float)
    prices_ugx_per_kg = np.asarray(prices_ugx_per_kg, dtype=float)
    if harvest_tonnes.ndim != 1 or prices_ugx_per_kg.ndim != 1:
        raise ValueError("harvest and prices must be 1D arrays")
    if len(harvest_tonnes) != len(prices_ugx_per_kg):
        raise ValueError("harvest and prices must have equal lengths")
    if np.any(harvest_tonnes < 0) or np.any(prices_ugx_per_kg < 0):
        raise ValueError("harvest and prices must be non-negative")
    return harvest_tonnes * 1000 * prices_ugx_per_kg

def revenue_summary(revenue: Iterable[float]) -> dict[str, float]:
    """Return descriptive statistics for a non-empty revenue series."""
    values = [float(x) for x in revenue]
    if not values:
        raise ValueError("revenue must not be empty")
    if any(x < 0 for x in values):
        raise ValueError("revenue must be non-negative")
    avg = mean(values)
    sd = stdev(values) if len(values) > 1 else 0.0
    return {
        "mean_UGX": avg,
        "median_UGX": median(values),
        "variance_UGX_squared": variance(values) if len(values) > 1 else 0.0,
        "std_dev_UGX": sd,
        "CV_percent": (sd / avg * 100) if avg else 0.0,
    }

weekly_revenue = revenue_from_harvest(harvest_10, prices)
summary = revenue_summary(weekly_revenue)
for key, value in summary.items():
    print(f"{key}: {value:,.2f}")

## 5. Risk classification and Monte Carlo 5% VaR

In [ ]:
@dataclass
class RiskAssessor:
    """Classify relative revenue risk using CV thresholds and estimate lower-tail VaR."""
    low_cv_percent: float = 10.0
    moderate_cv_percent: float = 20.0

    def __post_init__(self) -> None:
        if self.low_cv_percent < 0 or self.moderate_cv_percent <= self.low_cv_percent:
            raise ValueError("CV thresholds must be ordered and non-negative")

    def classify(self, revenue_values: Iterable[float]) -> str:
        """Classify revenue variability as Low, Moderate, or High."""
        stats = revenue_summary(revenue_values)
        cv = stats["CV_percent"]
        if cv < self.low_cv_percent:
            return "Low"
        if cv <= self.moderate_cv_percent:
            return "Moderate"
        return "High"

    def monte_carlo_var(self, harvest_tonnes: np.ndarray, price_model: PriceModel,
                        n_paths: int = 1000, seed: int = SEED) -> tuple[float, np.ndarray]:
        """Return 5% annual-revenue quantile (UGX) and all simulated annual revenues."""
        if n_paths < 1000:
            raise ValueError("n_paths must be at least 1000")
        rng = np.random.default_rng(seed)
        annual = np.empty(n_paths, dtype=float)
        for i in range(n_paths):
            path_seed = int(rng.integers(0, np.iinfo(np.int32).max))
            path_prices = price_model.simulate(len(harvest_tonnes), seed=path_seed)
            annual[i] = revenue_from_harvest(harvest_tonnes, path_prices).sum()
        return float(np.quantile(annual, 0.05)), annual

risk_assessor = RiskAssessor()
var_5, annual_revenues = risk_assessor.monte_carlo_var(harvest_10, price_model, n_paths=1000, seed=SEED)
annual_stats = revenue_summary(annual_revenues)
print(f"Monte Carlo paths: {len(annual_revenues):,}")
print(f"Mean annual revenue: {mean(annual_revenues):,.0f} UGX")
print(f"Median annual revenue: {median(annual_revenues):,.0f} UGX")
print(f"Annual revenue standard deviation: {stdev(annual_revenues):,.0f} UGX")
print(f"5% lower-tail annual revenue VaR: UGX {var_5:,.0f}")
print("Risk class for weekly revenue series:", risk_assessor.classify(weekly_revenue))
print("Risk class for simulated annual revenue:", risk_assessor.classify(annual_revenues))

## 6. Harvest-rate scenario analysis and theoretical MSY

The assignment gives the classical logistic-model reference `MSY = rK/4`. With `r = 0.4` and `K = 10,000`, this is 1,000 tonnes per time step. Here, however, the implemented harvest is **proportional to stock** (`h*N`) and `r` is interpreted as a weekly rate. The classic MSY formula assumes a constant-yield harvest from the logistic growth curve, so it is a useful reference but not directly interchangeable with annual total harvest in this proportional-harvest model. For a positive non-zero equilibrium under this model, `h` must be less than `r`; the equilibrium stock is `K*(1-h/r)`.

In [ ]:
HARVEST_RATES = [0.05, 0.10, 0.20, 0.30]
scenario_rows = []
scenario_data = {}

for h in HARVEST_RATES:
    fish = FishStock(harvest_rate=h)
    stock = fish.simulate(WEEKS)
    harvest = fish.weekly_harvest(stock)
    scenario_prices = price_model.simulate(WEEKS, seed=SEED + int(h * 1000))
    revenue = revenue_from_harvest(harvest, scenario_prices)
    scenario_rows.append({
        "h": h,
        "final_stock_tonnes": stock[-1],
        "total_harvest_tonnes": harvest.sum(),
        "total_revenue_UGX": revenue.sum(),
        "revenue_CV_percent": revenue_summary(revenue)["CV_percent"],
        "risk_class": risk_assessor.classify(revenue),
        "equilibrium_stock_tonnes_if_h_lt_r": K * (1 - h / R) if h < R else 0.0,
    })
    scenario_data[h] = {"stock": stock, "harvest": harvest, "revenue": revenue, "prices": scenario_prices}

MSY = R * K / 4
print(f"Theoretical MSY reference rK/4 = {MSY:,.0f} tonnes per model time step (classical logistic model)")
try:
    import pandas as pd
    scenario_table = pd.DataFrame(scenario_rows)
    display(scenario_table.style.format({
        "h": "{:.2f}", "final_stock_tonnes": "{:,.1f}", "total_harvest_tonnes": "{:,.1f}",
        "total_revenue_UGX": "UGX {:,.0f}", "revenue_CV_percent": "{:.2f}%",
        "equilibrium_stock_tonnes_if_h_lt_r": "{:,.1f}"
    }))
except ImportError:
    for row in scenario_rows:
        print(row)

## 7. Visualisations

Chart A compares stock trajectories under the four harvest rates. Chart B shows the simulated annual-revenue distribution and marks the 5% lower-tail VaR.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for h, data in scenario_data.items():
    ax.plot(np.arange(WEEKS + 1), data["stock"], label=f"h = {h:.2f}")
ax.set_title("Fish stock trajectories under alternative weekly harvest rates")
ax.set_xlabel("Week")
ax.set_ylabel("Fish stock (tonnes)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(annual_revenues / 1e9, bins=35, edgecolor="black")
ax.axvline(var_5 / 1e9, linestyle="--", linewidth=2, label=f"5% VaR = UGX {var_5/1e9:.2f} billion")
ax.set_title("Monte Carlo distribution of annual revenue (price risk only)")
ax.set_xlabel("Annual revenue (UGX billions)")
ax.set_ylabel("Number of simulations")
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 8. Extension: closed season for 8 weeks per year over 5 years

In [ ]:
YEARS = 5
CLOSED_WEEKS = 8
extension_fish = FishStock(harvest_rate=0.10)

stock_open, harvest_open = extension_fish.simulate_with_closed_season(years=YEARS, closed_weeks=0)
stock_closed, harvest_closed = extension_fish.simulate_with_closed_season(years=YEARS, closed_weeks=CLOSED_WEEKS)

extension_prices = price_model.simulate(YEARS * WEEKS, seed=SEED + 500)
revenue_open = revenue_from_harvest(harvest_open, extension_prices)
revenue_closed = revenue_from_harvest(harvest_closed, extension_prices)

extension_results = {
    "Open season": {
        "final_stock_tonnes": stock_open[-1],
        "five_year_harvest_tonnes": harvest_open.sum(),
        "five_year_revenue_UGX": revenue_open.sum(),
    },
    "8-week closed season": {
        "final_stock_tonnes": stock_closed[-1],
        "five_year_harvest_tonnes": harvest_closed.sum(),
        "five_year_revenue_UGX": revenue_closed.sum(),
    },
}
for label, result in extension_results.items():
    print(label)
    for metric, value in result.items():
        print(f"  {metric}: {value:,.1f}")

revenue_difference = revenue_closed.sum() - revenue_open.sum()
print(f"Revenue difference (closed minus open): UGX {revenue_difference:,.0f}")

## 9. Findings & Limitations

The Fibonacci baseline is not a suitable stock model because its growth is unbounded and does not reflect carrying capacity or harvest pressure. The logistic model is more defensible because stock growth slows near the 10,000-tonne carrying capacity and harvesting reduces the stock each week. In this implementation, harvest is a fraction of current stock, so the harvest-rate scenarios should be interpreted together with both final stock and cumulative harvest: a higher harvest rate may raise short-run removals while leaving a lower stock and weaker future growth. A non-zero equilibrium under this model requires `h < r`; rates approaching the weekly growth rate can substantially deplete stock.

Revenue is the product of harvested kilograms and a bounded, simulated UGX/kg price. The Monte Carlo 5% lower-tail revenue quantile summarizes price-driven downside under the chosen random-walk assumptions, not the probability of every possible business loss. The coefficient of variation is more interpretable than a fixed variance cutoff because it is dimensionless, but the Low/Moderate/High thresholds used here are illustrative and should be calibrated with the cooperative's historical revenue and risk tolerance.

Important limitations are that stock, carrying capacity, growth, harvest rates, price shocks, and price bounds are illustrative rather than estimated from local fisheries data. The model omits species composition, fish size and age, seasonal recruitment, illegal/unreported catch, weather, costs, exchange rates, export restrictions, and price responses to changing supply. Also, the classical `rK/4` MSY formula assumes a constant-yield logistic harvest; this notebook's `h*N` harvest rule differs, so the MSY figure is a theoretical reference rather than a directly comparable annual target. Real decisions should use locally validated stock assessments and economic data.

In [ ]:
# Fibonacci known prefix
assert fibonacci_numbers(7) == [0, 1, 1, 2, 3, 5, 8]

# No-harvest logistic trajectory stays within [0, K].
no_harvest_stock = FishStock(harvest_rate=0.0).simulate(52)
assert np.all(no_harvest_stock >= 0)
assert np.all(no_harvest_stock <= K + 1e-9)

# Price reproducibility and bounds
p1 = price_model.simulate(52, seed=123)
p2 = price_model.simulate(52, seed=123)
assert np.allclose(p1, p2)
assert np.all((p1 >= PRICE_MIN) & (p1 <= PRICE_MAX))

print("Notebook validation checks passed.")